---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Homework 3**: Improving Lexical Search

### 📅 **Due Date**: Day of Lecture 4, 11:59 PM


**Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

You'll apply what we covered in Lecture 3 (Lexical Search & BM25) to a real e-commerce search problem using the **WANDS dataset** 
- WANDS stands for Wayfair Annotated Dataset. It's a dataset of furniture products and search queries, along with human relevance judgments.

You will:
1. **Build a search engine** from scratch using BM25.
2. **Learn how to evaluate search results** using NDCG — a metric for measuring search quality
3. **Attempt to improve your search engine** by adding multiple fields
4. **Use LLMs to improve your search engine** by adding simple query understanding

Yes, *you* will do all these things. Let's go!

---

## Task 1: Environment Setup

First, let's set up your environment and verify everything works.

### 1a. Install dependencies and verify imports

Run `uv add pystemmer` in your terminal to add the Snowball stemmer. Then run the cell below to verify all imports work.

In [ ]:
print("hello")

In [ ]:
# Task 1a: Verify imports work
import pandas as pd
import numpy as np
from collections import Counter
import string
from pathlib import Path
# a stemmer from `pystemmer` for better tokenization
import Stemmer #library to do the stemming in NLPNLP
# llm packages
import litellm
from pydantic import BaseModel, Field
from typing import Optional
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Pandas display settings
# pd.set_option('display.max_colwidth', 100)

# Ignore pydantic warnings for litellm
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

print("All imports successful!")

### 1b. Verify API keys

Test that your API keys work by making a simple call.

In [ ]:
# Task 1b: Verify API keys
response = litellm.completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'API working!' and nothing else."}],
    max_tokens=20
)
print(response.choices[0].message.content)

---

## Task 2: Load and Explore the WANDS Dataset

The **WANDS dataset** (Wayfair Annotated Dataset) contains:
- 43K furniture products from Wayfair
- 480 real search queries
- 233K human relevance judgments (query-product pairs)

This is a real-world search benchmark used to evaluate e-commerce search systems!

**Data Source**: [WANDS on GitHub](https://github.com/wayfair/WANDS)

The data files are pre-downloaded in the `data/` directory:
- `wayfair-products.csv` - Product catalog
- `wayfair-queries.csv` - Search queries
- `wayfair-labels.csv` - Relevance judgments -- the bridge to judge ìf the product result is relevant to the query

### Data Loading Functions (provided)

Run the cell below to define the loading functions.

In [ ]:
# Data loading functions (provided)
# Note: Data from WANDS (Wayfair Annotated Dataset)
# Source: https://github.com/wayfair/WANDS

def load_wands_products(data_dir: str = "../data") -> pd.DataFrame: # -> pd.DataFrame - suggest type of return variable
    # "../data" means if no value put inside (), the default value is ../data - applies in prof folder where homework is not in the same hierarchy level as data folder.
    """
    Load WANDS products from local file.
    
    Args:
        data_dir: Path to the data directory containing wayfair-products.csv
        
    Returns:
        DataFrame with product information including product_id, product_name,
        product_class, category_hierarchy, product_description, etc.
    """
    filepath = Path(data_dir) / "wayfair-products.csv" #define path to csv file
    products = pd.read_csv(filepath, sep='\t') #create product dataframe
    products = products.rename(columns={'category hierarchy': 'category_hierarchy'}) #change column name, remove space, change to _
    return products

def load_wands_queries(data_dir: str = "../data") -> pd.DataFrame:
    """
    Load WANDS queries from local file.
    
    Args:
        data_dir: Path to the data directory containing wayfair-queries.csv
        
    Returns:
        DataFrame with query_id and query columns
    """
    filepath = Path(data_dir) / "wayfair-queries.csv"
    queries = pd.read_csv(filepath, sep='\t')
    return queries

def load_wands_labels(data_dir: str = "../data") -> pd.DataFrame:
    """
    Load WANDS relevance labels from local file.
    
    Args:
        data_dir: Path to the data directory containing wayfair-labels.csv
        
    Returns:
        DataFrame with query_id, product_id, label (Exact/Partial/Irrelevant),
        and grade (2/1/0) columns
    """
    filepath = Path(data_dir) / "wayfair-labels.csv"
    labels = pd.read_csv(filepath, sep='\t')
    grade_map = {'Exact': 2, 'Partial': 1, 'Irrelevant': 0}
    labels['grade'] = labels['label'].map(grade_map)
    return labels

print("Loading functions defined!")

### 2a. Load the data

Use the provided functions to load all three datasets. Print the number of rows in each.

In [ ]:
# Task 2a: Load the data

# YOUR CODE HERE
products = load_wands_products("data")
queries = load_wands_queries("data")
labels = load_wands_labels("data")

### 2b. Explore products

List the available columns, and display a few sample products. 

Which columns might be useful for search?

In [ ]:
# YOUR CODE HERE
print(products.columns)

In [ ]:
'''
The columns useful for search is product_name, product_description
'''

In [ ]:
# YOUR CODE HERE
products.head() #check df

In [ ]:
labels.head()

### 2c. Understand relevance judgments

The `labels` dataset contains human judgments of relevance. In particular, for each query-product pair, it contains:
| Label        | Grade | Description                                 |
|--------------|-------|---------------------------------------------|
| Exact        |   2   | This product is exactly what the user wants |
| Partial      |   1   | This product is somewhat relevant           |
| Irrelevant   |   0   | This product doesn't match the query        |

First, let's look at the distribution of grades.

In [ ]:
# Task 2c: Understand judgments

# YOUR CODE HERE
count_labels = Counter(labels['label'])
print(count_labels)

In [ ]:
# YOUR CODE HERE
print(f"Grade 2 (exact): {count_labels['Exact']}")
print(f"Grade 1 (partial):{count_labels['Partial']}")
print(f"Grade 0 (Irrelevant: {count_labels['Irrelevant']}")

---
## Task 3: Build and Run BM25 Search

Now let's build a BM25 search engine! We'll use the same concepts from Lecture 3.

### Provided Functions

We're giving you these functions to work with. Run the next cell to define them, then look at the examples.

| Function | What it does |
|----------|--------------|
| `snowball_tokenize(text)` | Tokenizes text, removes punctuation, stems words |
| `build_index(docs, tokenizer)` | Builds an inverted index from a list of documents |
| `get_tf(term, doc_id, index)` | Gets term frequency for a term in a document |
| `get_df(term, index)` | Gets document frequency for a term (how many docs contain the term) |
| `bm25_idf(df, num_docs)` | Calculates the IDF component of BM25 |
| `bm25_tf(tf, doc_len, avg_doc_len)` | Calculates the TF normalization for BM25 |
| `score_bm25(query, index, ...)` | Scores all documents for a query using BM25 |
| `search_products(query, ...)` | Searches and returns top-k results |

In [ ]:
# Provided functions - run this cell to define them

stemmer = Stemmer.Stemmer('english')
punct_trans = str.maketrans({key: ' ' for key in string.punctuation}) #replace punctuation with spaces, make it to translator that system can understand

def snowball_tokenize(text: str) -> list[str]:
    """
    Tokenize text with Snowball stemming.
    
    Args:
        text: The text to tokenize
        
    Returns:
        List of stemmed tokens
    """
    if pd.isna(text) or text is None:
        return []
    text = str(text).translate(punct_trans)
    tokens = text.lower().split()
    return [stemmer.stemWord(token) for token in tokens]

def build_index(docs: list[str], tokenizer) -> tuple[dict, list[int]]:
    """
    Build an inverted index from a list of documents.
    
    Args:
        docs: List of document strings to index
        tokenizer: Function that takes text and returns list of tokens
        
    Returns:
        index: dict mapping term -> {doc_id: term_count}
        doc_lengths: list of document lengths (in tokens)
    """
    index = {}
    doc_lengths = []
    
    for doc_id, doc in enumerate(docs):
        tokens = tokenizer(doc)
        doc_lengths.append(len(tokens))
        term_counts = Counter(tokens)
        
        for term, count in term_counts.items():
            if term not in index:
                index[term] = {}
            index[term][doc_id] = count
    
    return index, doc_lengths

def get_tf(term: str, doc_id: int, index: dict) -> int:
    """
    Get term frequency for a term in a document.
    
    Args:
        term: The term to look up
        doc_id: The document ID
        index: The inverted index
        
    Returns:
        Term frequency (count), or 0 if not found
    """
    if term in index and doc_id in index[term]:
        return index[term][doc_id]
    return 0

def get_df(term: str, index: dict) -> int:
    """
    Get document frequency for a term.
    
    Args:
        term: The term to look up
        index: The inverted index
        
    Returns:
        Number of documents containing the term
    """
    if term in index:
        return len(index[term])
    return 0

def bm25_idf(df: int, num_docs: int) -> float:
    """
    BM25 IDF formula.
    
    Args:
        df: Document frequency
        num_docs: Total number of documents
        
    Returns:
        IDF score
    """
    return np.log((num_docs - df + 0.5) / (df + 0.5) + 1)

def bm25_tf(tf: int, doc_len: int, avg_doc_len: float, k1: float = 1.2, b: float = 0.75) -> float:
    """
    BM25 TF normalization.
    
    Args:
        tf: Term frequency
        doc_len: Document length in tokens
        avg_doc_len: Average document length
        k1: Saturation parameter (default 1.2)
        b: Length normalization (default 0.75)
        
    Returns:
        Normalized TF score
    """
    return (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doc_len / avg_doc_len))

def score_bm25(query: str, index: dict, num_docs: int, doc_lengths: list[int], 
               tokenizer, k1: float = 1.2, b: float = 0.75) -> np.ndarray:
    """
    Score all documents using BM25.
    
    Args:
        query: The search query
        index: Inverted index
        num_docs: Total number of documents
        doc_lengths: List of document lengths
        tokenizer: Tokenization function
        
    Returns:
        Array of scores for each document
    """
    query_tokens = tokenizer(query)
    scores = np.zeros(num_docs)
    avg_doc_len = np.mean(doc_lengths) if doc_lengths else 1.0
    
    for token in query_tokens:
        df = get_df(token, index)
        if df == 0:
            continue
        
        idf = bm25_idf(df, num_docs)
        
        if token in index:
            for doc_id, tf in index[token].items():
                tf_norm = bm25_tf(tf, doc_lengths[doc_id], avg_doc_len, k1, b)
                scores[doc_id] += idf * tf_norm
    
    return scores

def search_products(query: str, products_df: pd.DataFrame, index: dict, 
                    doc_lengths: list[int], tokenizer, k: int = 10) -> pd.DataFrame:
    """
    Search products and return top-k results.
    
    Args:
        query: The search query
        products_df: DataFrame of products
        index: Inverted index
        doc_lengths: Document lengths
        tokenizer: Tokenization function
        k: Number of results to return
        
    Returns:
        DataFrame with top-k products and scores
    """
    scores = score_bm25(query, index, len(products_df), doc_lengths, tokenizer)
    top_k_idx = np.argsort(-scores)[:k]
    
    results = products_df.iloc[top_k_idx].copy()
    results['score'] = scores[top_k_idx]
    results['rank'] = range(1, k + 1)
    return results

print("All functions defined!")

In [ ]:
# Examples of each function

# 1. snowball_tokenize - tokenizes and stems text
print("1. snowball_tokenize('Running shoes are amazing!')")
print(f"   -> {snowball_tokenize('Running shoes are amazing!')}")
print("   Notice: 'Running' -> 'run', 'shoes' -> 'shoe', 'amazing' -> 'amaz'")

# 2. build_index - builds inverted index (we'll use a tiny example)
tiny_docs = ["red shoe", "blue shoe", "red hat"]
tiny_index, tiny_lengths = build_index(tiny_docs, snowball_tokenize)
print("\n2. build_index(['red shoe', 'blue shoe', 'red hat'], tokenizer)")
print(f"   Index: {tiny_index}")
print(f"   Lengths: {tiny_lengths}")

# 3. get_tf - get term frequency
print("\n3. get_tf('red', doc_id=0, tiny_index)")
print(f"   -> {get_tf('red', 0, tiny_index)}  (doc 0 = 'red shoe' has 1 'red')")

# 4. get_df - get document frequency  
print("\n4. get_df('red', tiny_index)")
print(f"   -> {get_df('red', tiny_index)}  ('red' appears in 2 documents)")

# 5. bm25_idf - calculate IDF (rare terms get higher scores)
print("\n5. bm25_idf(df=100, num_docs=10000)")
print(f"   -> {bm25_idf(100, 10000):.4f}  (term in 100 of 10000 docs)")

# 6. bm25_tf - normalize term frequency by document length
print("\n6. bm25_tf(tf=3, doc_len=50, avg_doc_len=100)")
print(f"   -> {bm25_tf(3, 50, 100):.4f}  (short doc gets boosted)")

# 7-8. score_bm25 and search_products - we'll use these next!
print("\nWe'll use score_bm25() and search_products() in Task 3a!")

### 3a. Create BM25 index for product_name

Build an inverted index for the `product_name` field and run a sample search for a product.

In [ ]:
# Task 3a: Create BM25 index for product_name

# YOUR CODE HERE
prod_name_raw=products['product_name'].fillna("").astype(str)
prod_index, prod_lengths = build_index(prod_name_raw,snowball_tokenize)
#print(f"   Index: {prod_index}")
#print(f"   Lengths: {prod_lengths}")


In [ ]:
query = "slow cooker"
num_docs=len(products['product_name'])
bm25_score= score_bm25(
    query=query, 
    index=prod_index, 
    num_docs=num_docs, 
    doc_lengths=prod_lengths, 
    tokenizer=snowball_tokenize
)
print(bm25_score) #expected return array

### 3b. Add product_description to search

Create a second index for `product_description` and combine scores from both fields.

**Hint**: You can combine the two scores by adding them together. This is like multi-field search from Lecture 3.

In [ ]:
# Task 3b: Add product_description to search

# YOUR CODE HERE
prod_des=products['product_description'].fillna("").astype(str)
prod_des_index, prod_des_lengths = build_index(prod_des,snowball_tokenize)
#print(f"   Index: {prod_index}")
#print(f"   Lengths: {prod_lengths}")

print(f"Product description index has {len(prod_des_index)} unique terms")
print(f"Product name index has {len(prod_des_index)} unique terms")


In [ ]:
def score_bm25f(query, indices, doc_lengths_dict, num_docs, tokenizer, k1=1.2, b=0.75):
    """
    BM25F scoring: blend TF across fields before applying IDF.
    
    Args:
        indices: dict of {field_name: index}
        doc_lengths_dict: dict of {field_name: [lengths]}
    """
    query_tokens = tokenizer(query)
    scores = np.zeros(num_docs)
    
    # Calculate average doc length across all fields
    total_lengths = np.zeros(num_docs)
    for field_lengths in doc_lengths_dict.values():
        total_lengths += np.array(field_lengths)
    avg_doc_len = np.mean(total_lengths)
    
    for token in query_tokens:
        # Calculate combined DF (max across fields)
        combined_df = 0
        for index in indices.values():
            combined_df = max(combined_df, get_df(token, index))
        
        if combined_df == 0:
            continue
        
        # Use combined DF for IDF
        idf = bm25_idf(combined_df, num_docs)
        
        for doc_id in range(num_docs):
            # Combine TF across fields (with length normalization per field)
            combined_impact = 0
            
            for field_name, index in indices.items():
                tf = get_tf(token, doc_id, index)
                if tf > 0:
                    doc_len = doc_lengths_dict[field_name][doc_id]
                    avg_field_len = np.mean(doc_lengths_dict[field_name])
                    # Length-normalized impact
                    impact = tf / (1 - b + b * doc_len / avg_field_len)
                    combined_impact += impact
            
            if combined_impact > 0:
                # Apply TF saturation to combined impact
                saturated = combined_impact / (combined_impact + k1)
                scores[doc_id] += idf * saturated
    
    return scores

In [ ]:
#Ref code from Lecture 3
indices = {'prod_name': prod_index,'prod_des':prod_des_index}
lengths = {'prod_name': prod_lengths,'prod_des':prod_des_lengths}
name_des = pd.DataFrame(products[['product_name','product_description']])
score_combine = score_bm25f(query,indices,lengths,len(name_des),snowball_tokenize)
print(score_combine)

---

## Task 4: Measuring Search Quality

We built a little search engine. How do we know if it's any good?

Consider two search results for "coffee table":

| Ranking A | Ranking B |
|-----------|-----------|
| 1. Wooden Coffee Table (Exact) | 1. Metal Lamp (Irrelevant) |
| 2. Glass Coffee Table (Exact) | 2. Wooden Coffee Table (Exact) |
| 3. Metal Lamp (Irrelevant) | 3. Glass Coffee Table (Exact) |

### A. Precision

One way to measure the quality of a ranking is to look at the precision within these first 3 results. 
- Precision is the ratio of relevant results to total results at position k.
- We call this precision@3, and more generally precision@k is the ratio of relevant results to total results at position k.
  
In this scenario, if we consider "exact" results as relevant, then both rankings have precision@3 = 2/3.

### B. DCG

Both rankings have the same precision, but Ranking A is clearly better 
- users look at results from the top down, and most people never scroll past the first few results
- as such, rankings that return relevant results earlier are better

So we need a metric that rewards **relevant** results, and rewards them **more** when they appear at the **top**

NDCG (Normalized Discounted Cumulative Gain) does this by giving each result a "gain" based on its relevance, then **discounting** that gain based on position.

**The formula** for each result at position $i$:

$$\text{gain}_i = \frac{2^{\text{relevance}} - 1}{\log_2(i + 1)}$$

- **Numerator** $(2^{\text{relevance}} - 1)$: How relevant is this result?
  - Irrelevant (0): $2^0 - 1 = 0$ (no gain)
  - Partial (1): $2^1 - 1 = 1$ (some gain)
  - Exact (2): $2^2 - 1 = 3$ (lots of gain!)
  
- **Denominator** $\log_2(i + 1)$: The "discount" based on position
  - Position 1: $\log_2(2) = 1$ (no discount)
  - Position 2: $\log_2(3) = 1.58$ (small discount)
  - Position 10: $\log_2(11) = 3.46$ (bigger discount)

**DCG** sums the discounted score for each result

$$\text{DCG} = \sum_{i=1}^{k} \frac{2^{\text{relevance}_i} - 1}{\log_2(i + 1)}$$

### 3. NDCG: Normalized DCG

One problem with DCG is that the score depends on how many relevant products exist. 
- A query with 10 exact matches will have a higher DCG than one with only 2, even if both rankings are "perfect."

One solution is to normalize by the *ideal* DCG — what the score would be if we ranked everything perfectly (all relevant results at the top).

$$\text{NDCG} = \frac{\text{DCG}}{\text{Ideal DCG}}$$

- **NDCG = 1.0**: Perfect -- best possible order
- **NDCG = 0.5**: OK -- some good some bad
- **NDCG = 0.0**: Worst -- results are irrelevant

**Read the above carefully.** In the next cell, explain in your own words: why does the discount formula use $\log_2$? What happens to results at position 1 vs position 10?

In [ ]:
# Task 4a: Answer in a comment
# Why does DCG use log2 for the discount? What's the effect on position 1 vs position 10?
#
# YOUR ANSWER HERE
#Because log2 has positive correlation with position. The higher the position will lead to bigger denominator, therefore bigger discount on the result. 
#It also decreases the score more smoothly and take into consideration the position of the search result. (Result rank 2nd would be more important than rank 10th, but rank 10th and rank 15th does not make much difference)
#Therefore log2 is a proper calculation to balance the penalty and still value the position of search result.

### 4b. Calculate NDCG by hand

Let's work through an example step by step.

**Scenario**: You search for "wooden coffee table" and get these results:

| Position | Product | Relevance |
|----------|---------|----------|
| 1 | Glass Coffee Table | Partial (1) |
| 2 | Wooden Coffee Table | Exact (2) |
| 3 | Wooden Side Table | Partial (1) |
| 4 | Metal Coffee Table | Irrelevant (0) |
| 5 | Wooden Coffee Table (different) | Exact (2) |

**Your task**: Calculate DCG and NDCG@5 by hand.

In [ ]:
# Task 4b: Calculate NDCG by hand

# YOUR CODE HERE
import math

relevant_score=[1,2,1,0,2] #for small data
dcg_total = 0
for p in range(1,6):
    dcg = (2**(relevant_score[p-1])-1)/math.log2(p+1)
    dcg_total += dcg
print(f"DCG score is {dcg_total:,.4f}")

### 4c. Implement NDCG function

Now implement the NDCG calculation in code. Verify your implementation matches your hand calculation!

In [ ]:
# Task 4c: Implement NDCG function

# YOUR CODE HERE
i_relevant = sorted(relevant_score,reverse=True) #ideal is the relevance from highest to lowest
i_dcg_total = 0
for p in range(1,6):
    i_dcg = (2**(i_relevant[p-1])-1)/math.log2(p+1)
    i_dcg_total += i_dcg
n_dcg = dcg_total/i_dcg_total
print(f"NDCG Score is {n_dcg:,.4f}")

In [ ]:
def calculate_dcg(relevance, k= 5):
    rel = list(relevance)
    k_eff = min(k, len(rel))
    dcg_total = 0.0
    for p in range(1, k_eff + 1):
        dcg_total += (2**rel[p-1] - 1) / math.log2(p + 1)
    return dcg_total

In [ ]:
def calculate_ndcg(relevance, k=5):
    rel = list(relevance)
    k_eff = min(k, len(rel))
    if k_eff == 0:
        return 0.0

    dcg_total = calculate_dcg(rel, k=k_eff)

    i_relevant = sorted(rel, reverse=True)  
    idcg_total = 0.0
    for p in range(1, k_eff + 1):
        idcg_total += (2**i_relevant[p-1] - 1) / math.log2(p + 1)

    if idcg_total == 0:
        return 0.0

    return dcg_total / idcg_total

In [ ]:
# Verify your implementation matches your hand calculation
test_relevances = [1, 2, 1, 0, 2]

dcg = calculate_dcg(test_relevances, k=5)
n_dcg = calculate_ndcg(test_relevances, k=5)

print(f"DCG@5 = {dcg:.4f}")
print(f"NDCG@5 = {n_dcg:.4f}")

# These should match your hand calculations from Task 4b!

---

## Task 5: Evaluate Your Search Strategy

Now let's evaluate our BM25 search across all queries in the WANDS dataset.

### Evaluation Helper Functions (provided)

In [ ]:
# Evaluation helper functions (provided)

def get_relevance_grades(product_ids: list[int], query_id: int, labels_df: pd.DataFrame) -> list[int]:
    """
    Get relevance grades for a list of product IDs given a query.
    
    Args:
        product_ids: List of product IDs in rank order
        query_id: The query ID
        labels_df: DataFrame with relevance labels
        
    Returns:
        List of relevance grades (0, 1, or 2) for each product
    """
    # Get labels for this query
    query_labels = labels_df[labels_df['query_id'] == query_id] #defined and mapped in the beginning
    label_dict = dict(zip(query_labels['product_id'], query_labels['grade']))
    
    # Look up grades for each product (default to 0 if not labeled)
    return [label_dict.get(pid, 0) for pid in product_ids]

def evaluate_single_query(query_text: str, query_id: int, products_df: pd.DataFrame,
                          labels_df: pd.DataFrame, search_func, k: int = 10) -> float:
    """
    Evaluate search for a single query.
    
    Args:
        query_text: The search query text
        query_id: The query ID for looking up labels
        products_df: DataFrame of products
        labels_df: DataFrame with relevance labels
        search_func: Function that takes query and returns DataFrame with product_id column
        k: Number of results to consider
        
    Returns:
        NDCG@k score for this query
    """
    results = search_func(query_text)
    product_ids = results['product_id'].tolist()[:k]
    relevances = get_relevance_grades(product_ids, query_id, labels_df)
    return calculate_ndcg(relevances, k)

def evaluate_search(search_func, products_df: pd.DataFrame, queries_df: pd.DataFrame,
                    labels_df: pd.DataFrame, k: int = 10, verbose: bool = True) -> pd.DataFrame:
    """
    Evaluate search across all queries.
    
    Args:
        search_func: Function that takes query string and returns DataFrame with product_id
        products_df: DataFrame of products
        queries_df: DataFrame of queries
        labels_df: DataFrame with relevance labels
        k: Number of results to consider
        verbose: Whether to print progress
        
    Returns:
        DataFrame with query_id, query, and ndcg columns
    """
    results = []
    
    for _, row in queries_df.iterrows():
        query_id = row['query_id']
        query_text = row['query']
        
        ndcg = evaluate_single_query(query_text, query_id, products_df, 
                                     labels_df, search_func, k)
        results.append({
            'query_id': query_id,
            'query': query_text,
            'ndcg': ndcg
        })
    
    results_df = pd.DataFrame(results)
    
    if verbose:
        print(f"Evaluated {len(results_df)} queries")
        print(f"Mean NDCG@{k}: {results_df['ndcg'].mean():.4f}")
    
    return results_df

print("Evaluation functions defined!")

### 5a. Run evaluation on all queries

Create a search function and evaluate it on all queries.

In [ ]:
queries.head(10)

In [ ]:
query_top10 = queries['query'].iloc[:10]
print(query_top10)


In [ ]:
# Task 5a: Run evaluation on all queries

_docid_to_productid = products["product_id"].tolist()
_num_docs = len(_docid_to_productid)

def bm25_search_FINAL(query_text: str, topn: int = 20) -> pd.DataFrame:
    scores = score_bm25(
        query=query_text,
        index=prod_index,
        num_docs=_num_docs,
        doc_lengths=prod_lengths,
        tokenizer=snowball_tokenize,
        k1=1.2,
        b=0.75
    )

    scores = np.asarray(scores, dtype=float)  # ép kiểu để tránh dict/object
    top_doc_ids = np.argsort(-scores)[:topn]

    return pd.DataFrame({
        "product_id": [_docid_to_productid[int(i)] for i in top_doc_ids],
        "score": [float(scores[int(i)]) for i in top_doc_ids]
    })

def search_func(query_text: str) -> pd.DataFrame:
    return bm25_search_FINAL(query_text, topn=20)

# print first query evaluation for testing
print(search_func(queries.iloc[0]["query"]).head())

evaluation_results = evaluate_search(
    search_func=search_func,
    products_df=products,
    queries_df=queries,
    labels_df=labels,
    k=10,
    verbose=True
)

### 5b. Identify failing queries

Find queries where our search performed poorly (NDCG = 0 or very low). Analyze one of them.

In [ ]:
# Task 5b: Identify failing queries

# YOUR CODE HERE
fail_queries = evaluation_results[evaluation_results['ndcg']<0.1]
print(fail_queries)
print(len(fail_queries))

#define fail query to test
f_query = fail_queries['query'].iloc[2]
print(f_query)



In [ ]:
search_func(f_query).head(10)


Let's take "medium size chandelier" as an example. Despite having bm25 score for the top 10 results, it has ndcg = 0. -> Among the top 10 results, there are actually no matching product with the query based on ground truth ('labels' file), even though there might be matching keywords with the product names.

### 5c. Analyze the distribution

Visualize the distribution of NDCG scores.

In [ ]:
# Task 5c: Analyze the distribution

# YOUR CODE HERE
import matplotlib.pyplot as plt

ndcg_scores = evaluation_results['ndcg']

plt.figure()
plt.hist(ndcg_scores, bins=30)
plt.xlabel("NDCG@10")
plt.ylabel("Number of queries")
plt.title("Distribution of NDCG Scores")
plt.show()

---

## Task 6: Improve Search with Additional Fields

Our baseline only searches the `product_name` field. Let's improve by adding more fields!

### 6a. Index product_class field

The `product_class` field contains the category of the product (e.g., "Rugs", "Coffee Tables"). This is a powerful signal!

Create a search function that combines all three fields (name, description, class).

In [ ]:
# Task 6a: Index product_class field

# YOUR CODE HERE
prod_class=products['product_class'].fillna("").astype(str)
prod_class_index, prod_class_lengths = build_index(prod_class, snowball_tokenize)
print(f"Product description index has {len(prod_class_index)} unique terms")
print(f"Product name index has {len(prod_class_index)} unique terms")


In [ ]:
prod_des=products['product_description'].fillna("").astype(str)
prod_des_index, prod_des_lengths = build_index(prod_des,snowball_tokenize)
#print(f"   Index: {prod_index}")
#print(f"   Lengths: {prod_lengths}")

print(f"Product description index has {len(prod_des_index)} unique terms")
print(f"Product name index has {len(prod_index)} unique terms")


In [ ]:
indices_3={'prod_name': prod_index,'prod_des':prod_des_index,'prod_class':prod_class_index}
lengths_3 = {'prod_name': prod_lengths,'prod_des':prod_des_lengths,'prod_class':prod_class_lengths}
name_des_class=pd.DataFrame(products[['product_name','product_description','product_class']])
score_combine3=score_bm25f(query,indices,lengths,len(name_des_class),snowball_tokenize)
print(score_combine3)

In [ ]:
#search function combine 3

doc_to_prod = products["product_id"].tolist()
numdocs = len(_docid_to_productid)

def bm25_search_combined(query_text: str, topn: int = 20) -> pd.DataFrame:
    scores = score_bm25f(
        query=query_text,
        indices=indices_3,
        num_docs=numdocs,
        doc_lengths_dict=lengths_3,
        tokenizer=snowball_tokenize,
        k1=1.2,
        b=0.75
    )

    scores = np.asarray(scores, dtype=float)  # ép kiểu để tránh dict/object
    top_doc_ids = np.argsort(-scores)[:topn]

    return pd.DataFrame({
        "product_id": [doc_to_prod[int(i)] for i in top_doc_ids],
        "score": [float(scores[int(i)]) for i in top_doc_ids]
    })

def search_func_3(query_text: str) -> pd.DataFrame:
    return bm25_search_combined(query_text, topn=20)

### 6b. Evaluate three-field search

Now evaluate your three-field search on all queries to see how it compares to the baseline.

In [ ]:
# Task 6b: Evaluate three-field search

# YOUR CODE HERE

#here i only take example of 5 queries because all queries take too much time. 

query_top5 = queries.iloc[:5]

print(search_func_3(queries.iloc[0]["query"]).head())

evaluation_results_3 = evaluate_search(
    search_func=search_func_3,
    products_df=products,
    queries_df=query_top5,
    labels_df=labels,
    k=10,
    verbose=True
)

### 6c. Compare to baseline

Analyze which queries improved and which degraded when using three-field search.

In [ ]:
evaluation_results_base = evaluate_search(
    search_func=search_func,
    products_df=products,
    queries_df=query_top5, #change to top 5 query so that both result are from the same queries => fair to compare
    labels_df=labels,
    k=10,
    verbose=True
)

In [ ]:
# Task 6c: Compare to baseline

# YOUR CODE HERE
# pick only needed columns
# pick only needed columns
base = (
    evaluation_results_base[['query_id', 'query', 'ndcg']]
    .rename(columns={'ndcg': 'ndcg_base'})
)

new = (
    evaluation_results_3[['query_id', 'query', 'ndcg']]
    .rename(columns={'ndcg': 'ndcg_3field'})
)

cmp = base.merge(new, on=['query_id', 'query'], how='inner')
cmp['delta'] = cmp['ndcg_3field'] - cmp['ndcg_base']

# summary
print("Total queries compared:", len(cmp))
print("Improved:", (cmp['delta'] > 0).sum())
print("Degraded:", (cmp['delta'] < 0).sum())
print("Unchanged:", (cmp['delta'] == 0).sum())

# top improvements / degradations
print("\nTop +delta (improved):")
print(
    cmp.sort_values('delta', ascending=False).head(15)[
        ['query_id', 'query', 'ndcg_base', 'ndcg_3field', 'delta']
    ]
)

print("\nTop -delta (degraded):")
print(
    cmp.sort_values('delta', ascending=True).head(15)[
        ['query_id', 'query', 'ndcg_base', 'ndcg_3field', 'delta']
    ]
)



---

## Task 7: Query Understanding with LLM

Sometimes users search for "star wars rug" when they really want a "rug with Star Wars theme". An LLM can help us understand what the user is actually looking for!

### 7a. Extract product type from query

Write a function using LiteLLM with structured outputs (Pydantic) to extract key information from a query.

In [ ]:
# Task 7a: Extract product type, theme, material, color, and any other information you deem relevcant from the query

# YOUR CODE HERE
from pydantic import BaseModel, Field

class Product(BaseModel):
    product_type: str = Field(description="Product type")
    theme: str | None = Field(default = None, description="Product theme")
    material: str = Field(description='Product material')
    color: str = Field(description="Product Color")

    pass

def extract_prod_result(queries: str) -> Product:
    result_q = litellm.completion(
        model="gpt-5-mini",
        messages=[{"role":"user","content":queries}],
        response_format = Product
    )
    return Product.model_validate_json(result_q.choices[0].message.content)
    pass

In [ ]:
# Test your query understanding function by running it against these test queries
test_queries = [
    "star wars rug",
    "wooden coffee table",
    "blue leather sofa",
    "modern metal bookshelf"
]

for q in test_queries:
    # YOUR CODE HERE
    prod_result = extract_prod_result(q)
    print(prod_result)
    pass

### 7b. Create an LLM-enhanced search

Use the extracted product type to boost matching results. If the LLM identifies "rug" as the product type, boost products where `product_class` contains "rug".

In [ ]:
# Task 7b: Create an LLM-enhanced search

# YOUR CODE HERE
def search_func_llm_boost(query_text: str, topn: int = 20) -> pd.DataFrame:
    # baseline search
    results = search_func(query_text).copy()   # BM25 baseline

    #extract structured info from LLM
    prod_info = extract_prod_result(query_text)

    if not prod_info.product_type:
        return results.head(topn)

    product_type = prod_info.product_type.lower()

    # join product_class
    results = results.merge(
        products[["product_id", "product_class"]],
        on="product_id",
        how="left"
    )

    BOOST = 1.3 
    mask = results["product_class"].str.lower().str.contains(product_type, na=False)

    results.loc[mask, "score"] *= BOOST

    results = results.sort_values("score", ascending=False)

    return results.head(topn)



In [ ]:
# YOUR CODE HERE

print("Baseline:")
display(search_func(f_query).head(5))

print("\nLLM boosted:")
display(search_func_llm_boost(f_query).head(5))


---
## Task 8: Submit via Pull Request

Now let's submit your work using the Git workflow from previous homeworks.
- [ ] Create a new branch called `homework-3`
- [ ] Commit you work and push it to the branch
- [ ] Create a PR with a nice description of your changes
- [ ] Merge the PR to your main branch
  
**The TA will verify your submission by checking the merged PR on your GitHub repo.**

**Also remember to submit your homework on Blackboard!**
